<a href="https://colab.research.google.com/github/evkoff/DI-Bootcamp-Stage1/blob/main/Week11/Day2/DailyChallenge/W11D2_Stock_Price_Prediction_with_LStm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Stock Price Prediction with LSTM


# 👩‍🏫 👩🏿‍🏫 What You’ll learn
How to preprocess and prepare time-series data for machine learning models.
How to build and train an LSTM (Long Short-Term Memory) model using PyTorch.
How to evaluate the performance of a regression model using metrics like R².


# 🛠️ What you will create
A preprocessed dataset for stock price prediction.
A trained LSTM model to predict future stock prices.


# Understanding PyTorch
PyTorch is an open-source machine learning framework based on the Torch library, used for applications such as computer vision and natural language processing. It’s known for its flexibility and ease of use, making it popular for both research and production.

# Key PyTorch Functions You’ll Use:

torch.nn.Module: Base class for all neural network modules. You’ll use this to define your LSTM model.
torch.nn.LSTM: Implements a Long Short-Term Memory (LSTM) network.
torch.nn.Linear: Applies a linear transformation to the incoming data (i.e., a fully connected layer).
torch.nn.Dropout: Applies dropout regularization to prevent overfitting.
torch.optim.Adam: Implements the Adam optimization algorithm.
torch.nn.MSELoss: Implements the Mean Squared Error loss function.
torch.utils.data.Dataset: An abstract class representing a dataset.
torch.utils.data.DataLoader: Combines a dataset and a sampler, and provides single- or multi-process iterators over the dataset.
torch.Tensor: A multi-dimensional matrix containing elements of a single data type.
torch.save and torch.load: used to save and load trained models.

# What You Need to Do
1. Install Required Libraries

Ensure you have the necessary libraries installed, including gensim, spacy, torch, and scikit-learn.

2. Load and Preprocess the Dataset

Download the stock market dataset.
Drop unnecessary columns and create a target column for the next day’s closing price.
Normalize the dataset using MinMaxScaler.

3. Prepare the Dataset for Training

Split the dataset into training, validation, and testing sets.
Create a custom PyTorch Dataset class to handle the data.
Use DataLoader to create iterable datasets for training and evaluation.

4. Define the LSTM Model

Create an LSTM model using PyTorch.
Define the model architecture, including GRU layers, dropout, and a dense layer.

5. Train the Model

Set up the optimizer and loss function.
Implement training and validation loops.
Train the model for a specified number of epochs.

6. Evaluate the Model

Calculate the R² score to evaluate the model’s performance on the test set.
Save the scaler object for future predictions.

In [1]:
# Data manipulation
import numpy as np
import pandas as pd

# Data preprocessing
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score

# PyTorch
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Visualization
import matplotlib.pyplot as plt

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

In [2]:
import pandas as pd

df = pd.read_csv("AAPL.csv")

df.head()

,Date,Open,High,Low,Close,Adj Close,Volume
0,1980-12-12,0.513393,0.515625,0.513393,0.513393,0.406782,117258400
1,1980-12-15,0.488839,0.488839,0.486607,0.486607,0.385558,43971200
2,1980-12-16,0.453125,0.453125,0.450893,0.450893,0.357260,26432000
3,1980-12-17,0.462054,0.464286,0.462054,0.462054,0.366103,21610400
4,1980-12-18,0.475446,0.477679,0.475446,0.475446,0.376715,18362400


In [3]:
# Display general information about the dataset
df.info()

# Display descriptive statistics
df.describe()

# Check for missing values
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9909 entries, 0 to 9908
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Date       9909 non-null   object 
 1   Open       9909 non-null   float64
 2   High       9909 non-null   float64
 3   Low        9909 non-null   float64
 4   Close      9909 non-null   float64
 5   Adj Close  9909 non-null   float64
 6   Volume     9909 non-null   int64  
dtypes: float64(5), int64(1), object(1)
memory usage: 542.0+ KB


,0
Date,0
Open,0
High,0
Low,0
Close,0
Adj Close,0
Volume,0


In [4]:
# Convert the Date column to datetime format
df["Date"] = pd.to_datetime(df["Date"])

# Create the target column (next day's closing price)
df["Target"] = df["Close"].shift(-1)

# Remove the last row because it has no target value
df = df.dropna().reset_index(drop=True)

# Display the first rows
df.head()

,Date,Open,High,Low,Close,Adj Close,Volume,Target
0,1980-12-12,0.513393,0.515625,0.513393,0.513393,0.406782,117258400,0.486607
1,1980-12-15,0.488839,0.488839,0.486607,0.486607,0.385558,43971200,0.450893
2,1980-12-16,0.453125,0.453125,0.450893,0.450893,0.357260,26432000,0.462054
3,1980-12-17,0.462054,0.464286,0.462054,0.462054,0.366103,21610400,0.475446
4,1980-12-18,0.475446,0.477679,0.475446,0.475446,0.376715,18362400,0.504464


In [5]:
# Select features for the model
feature_cols = ["Open", "High", "Low", "Close", "Volume"]

# Create feature matrix (X) and target vector (y)
X = df[feature_cols]
y = df["Target"]

# Display the first rows
print(X.head())
print(y.head())

       Open      High       Low     Close     Volume
0  0.513393  0.515625  0.513393  0.513393  117258400
1  0.488839  0.488839  0.486607  0.486607   43971200
2  0.453125  0.453125  0.450893  0.450893   26432000
3  0.462054  0.464286  0.462054  0.462054   21610400
4  0.475446  0.477679  0.475446  0.475446   18362400
0    0.486607
1    0.450893
2    0.462054
3    0.475446
4    0.504464
Name: Target, dtype: float64


In [6]:
# Define split indices
train_size = int(len(X) * 0.7)
val_size = int(len(X) * 0.15)

# Split features
X_train = X[:train_size]
X_val = X[train_size:train_size + val_size]
X_test = X[train_size + val_size:]

# Split targets
y_train = y[:train_size]
y_val = y[train_size:train_size + val_size]
y_test = y[train_size + val_size:]

# Print dataset sizes
print(f"Train: {len(X_train)}")
print(f"Validation: {len(X_val)}")
print(f"Test: {len(X_test)}")

Train: 6935
Validation: 1486
Test: 1487


In [7]:
# Initialize the scaler
scaler = MinMaxScaler()

# Scale the training features
X_train_scaled = scaler.fit_transform(X_train) #result will be NumPy array

# Scale the validation and test features
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [8]:
# Convert scaled arrays back to DataFrames for easier inspection
X_train_scaled = pd.DataFrame(X_train_scaled, columns=feature_cols)
X_val_scaled = pd.DataFrame(X_val_scaled, columns=feature_cols)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=feature_cols)

# Display the first rows of the scaled training data
X_train_scaled.head()

,Open,High,Low,Close,Volume
0,0.011060,0.011007,0.011296,0.011180,0.063023
1,0.010197,0.010077,0.010341,0.010235,0.023516
2,0.008942,0.008837,0.009068,0.008976,0.014061
3,0.009256,0.009224,0.009466,0.009369,0.011462
4,0.009726,0.009690,0.009943,0.009842,0.009711


In [9]:
# Function to create sequences for LSTM
def create_sequences(features, targets, sequence_length):
    X_seq = []
    y_seq = []

    for i in range(len(features) - sequence_length):
        X_seq.append(features[i:i + sequence_length])
        y_seq.append(targets.iloc[i + sequence_length])

    return np.array(X_seq), np.array(y_seq)

In [10]:
# Number of previous days used for prediction
sequence_length = 30

# Create sequences
X_train_seq, y_train_seq = create_sequences(
    X_train_scaled,
    y_train,
    sequence_length
)

X_val_seq, y_val_seq = create_sequences(
    X_val_scaled,
    y_val,
    sequence_length
)

X_test_seq, y_test_seq = create_sequences(
    X_test_scaled,
    y_test,
    sequence_length
)

# Check the shapes
print(X_train_seq.shape) #number of examples (observations), number of days in the window (sequence length), number of features,
print(y_train_seq.shape)
print(X_train_seq[0].shape)

(6905, 30, 5)
(6905,)
(30, 5)


Progress:
✅ Import libraries

✅ Load dataset
✅ EDA
✅ Create target
✅ Feature selection
✅ Train / Validation / Test split
✅ Scaling
✅ Create sequences
⬜ Custom Dataset
⬜ DataLoader
⬜ Build LSTM
⬜ Training
⬜ Evaluation
⬜ Save scaler

In [11]:
# preparation of the data for neural network - one dataset instead of 2 arrays

# Custom Dataset class for stock price prediction
class StockDataset(Dataset):

    def __init__(self, features, targets):
        self.X = torch.FloatTensor(features) #data will be in the form of Torch Tensor (before they were NumPy), FloatTensor (not IntTensor) - in order to store float numbers, because weights are float
        self.y = torch.FloatTensor(targets)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [12]:
# Create Dataset objects
train_dataset = StockDataset(X_train_seq, y_train_seq)
val_dataset = StockDataset(X_val_seq, y_val_seq)
test_dataset = StockDataset(X_test_seq, y_test_seq)

# Check the number of training samples
print(len(train_dataset))

6905


In [13]:
# Create Dataloaders - it will say "I will be giving to a modfel 32 or 64 samples at a time (batch)."
batch_size = 32 #how many sequences is in the batch

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True #shuffle=True changes the order of the sequences.
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

In [14]:
# Inspect one training batch
X_batch, y_batch = next(iter(train_loader))

print(X_batch.shape)
print(y_batch.shape)

torch.Size([32, 30, 5])
torch.Size([32])


In [15]:
# Define the LSTM model
# nn. Module - all models in PyTorch are inherited from it.

class StockLSTM(nn.Module):

    def __init__(self, input_size, hidden_size, num_layers, dropout):
        super().__init__()

        # LSTM layer
        self.lstm = nn.LSTM(
            input_size=input_size, # number of internal neurons in LSTM.
            hidden_size=hidden_size, # size of the memory to remember previous days
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )

        # Dropout layer
        self.dropout = nn.Dropout(dropout)

        # Output layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):

        # Pass data through the LSTM layer
        output, (hidden, cell) = self.lstm(x)

        # Take the output from the last time step
        x = hidden[-1]

        # Apply dropout
        x = self.dropout(x)

        # Generate the prediction
        x = self.fc(x)

        return x

In [22]:
# Initialize the model
model = StockLSTM(
    input_size=5,
    hidden_size=64,
    num_layers=2,
    dropout=0.2
)

print(model)

StockLSTM(
  (lstm): LSTM(5, 64, num_layers=2, batch_first=True, dropout=0.2)
  (dropout): Dropout(p=0.2, inplace=False)
  (fc): Linear(in_features=64, out_features=1, bias=True)
)


In [23]:
# Define the loss function and optimizer
criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [24]:
# model.train()

# optimizer.zero_grad()

# predictions = model(...)

# loss = criterion(...)

# loss.backward()

# optimizer.step()


# Get one batch from the training DataLoader
X_batch, y_batch = next(iter(train_loader))

print(X_batch.shape)
print(y_batch.shape)

torch.Size([32, 30, 5])
torch.Size([32])


In [25]:
# Run one forward pass
predictions = model(X_batch)

print(predictions.shape)
print(predictions[:5])

torch.Size([32, 1])
tensor([[-0.0167],
        [-0.0280],
        [-0.0023],
        [-0.0334],
        [-0.0084]], grad_fn=<SliceBackward0>)


In [26]:
# Remove the extra dimension from predictions
predictions = predictions.squeeze()

# Calculate the loss
loss = criterion(predictions, y_batch)

print(loss)

tensor(21.7765, grad_fn=<MseLossBackward0>)


In [21]:
# # Demonstration of a single training epoch (used for learning purposes)
# # Iterate through all training batches (one epoch) - so update weights of the model
# for X_batch, y_batch in train_loader:

#     # Clear gradients from the previous iteration
#     optimizer.zero_grad()

#     # Perform a forward pass through the model
#     predictions = model(X_batch)

#     # Remove the extra dimension from the output
#     predictions = predictions.squeeze()

#     # Compute the loss between predictions and true values
#     loss = criterion(predictions, y_batch)

#     # Compute gradients using backpropagation
#     loss.backward()

#     # Update model parameters
#     optimizer.step()

In [27]:
# Set the number of training epochs
num_epochs = 20

# Store the training loss for each epoch
train_losses = []

# Start the training loop
for epoch in range(num_epochs):

    # Switch the model to training mode
    model.train()

    # Initialize the running loss for the current epoch, loss will be calculated in the beginning of each epoch
    running_loss = 0.0

    # Iterate through all training batches
    for X_batch, y_batch in train_loader:

        # Clear gradients from the previous iteration
        optimizer.zero_grad()

        # Perform a forward pass
        predictions = model(X_batch)

        # Remove the extra dimension from the output
        predictions = predictions.squeeze()

        # Compute the loss
        loss = criterion(predictions, y_batch)

        # Compute gradients
        loss.backward()

        # Update model parameters
        optimizer.step()

        # Accumulate the loss
        running_loss += loss.item() # Accumulate batch losses for the current epoch

    # Compute the average loss for the epoch
    epoch_loss = running_loss / len(train_loader)

    # Save the epoch loss
    train_losses.append(epoch_loss)

    # Print the training progress
    print(f"Epoch {epoch + 1}/{num_epochs}, Training Loss: {epoch_loss:.4f}")

Epoch 1/20, Training Loss: 9.5690
Epoch 2/20, Training Loss: 1.8295
Epoch 3/20, Training Loss: 0.8153
Epoch 4/20, Training Loss: 0.5053
Epoch 5/20, Training Loss: 0.3574
Epoch 6/20, Training Loss: 0.3012
Epoch 7/20, Training Loss: 0.3095
Epoch 8/20, Training Loss: 0.2783
Epoch 9/20, Training Loss: 0.2943
Epoch 10/20, Training Loss: 0.2505
Epoch 11/20, Training Loss: 0.2532
Epoch 12/20, Training Loss: 0.2539
Epoch 13/20, Training Loss: 0.2225
Epoch 14/20, Training Loss: 0.2489
Epoch 15/20, Training Loss: 0.2092
Epoch 16/20, Training Loss: 0.2371
Epoch 17/20, Training Loss: 0.2230
Epoch 18/20, Training Loss: 0.2250
Epoch 19/20, Training Loss: 0.2153
Epoch 20/20, Training Loss: 0.2324


In [35]:
# Switch the model to evaluation mode
model.eval()

# Initialize the validation loss
val_loss = 0.0

# Disable gradient computation during evaluation
with torch.no_grad():

    # Iterate through all validation batches
    for X_batch, y_batch in val_loader:

        # Perform a forward pass
        predictions = model(X_batch)

        # Remove the extra dimension from the output
        predictions = predictions.squeeze()

        # Compute the validation loss
        loss = criterion(predictions, y_batch)

        # Accumulate the validation loss
        val_loss += loss.item()

# Compute the average validation loss
val_loss /= len(val_loader)

# Display the validation loss
print(f"Validation Loss: {val_loss:.4f}")

Validation Loss: 1070.1341


The validation loss is significantly higher than the training loss because the model is trained on the earlier part of Apple's historical data, while the validation set contains much higher stock prices from later years. This distribution shift makes generalization more difficult.

In [36]:
# Import the R² metric
from sklearn.metrics import r2_score

# Switch the model to evaluation mode
model.eval()

# Lists to store predictions and true values
predictions_list = []
targets_list = []

# Disable gradient computation during evaluation
with torch.no_grad():

    # Iterate through all test batches
    for X_batch, y_batch in test_loader:

        # Perform a forward pass
        predictions = model(X_batch)

        # Remove the extra dimension from the output
        predictions = predictions.squeeze()

        # Store predictions
        predictions_list.extend(predictions.cpu().numpy())

        # Store true values
        targets_list.extend(y_batch.cpu().numpy())

# Calculate the R² score
r2 = r2_score(targets_list, predictions_list)

# Display the result
print(f"R² Score: {r2:.4f}")

R² Score: -5.7886


In [37]:
# Import the joblib library
import joblib

# Save the fitted scaler
joblib.dump(scaler, "scaler.pkl")

# Confirm that the scaler has been saved
print("Scaler saved successfully.")

Scaler saved successfully.
